# Qwen3.5-9B · P0~P5 추론 프롬프트 비교

**선택 반영 v1.1:** 학습640/추론672, answer_only. 보고된 기준 성능 94.0%(470/500), full_text 대비 +2.2%p.

**학습 640²인 동일 LoRA / 추론 672² / 동일 검증 약 500개**에서 사용자 메시지의 마지막 지시만 바꿉니다.
시스템 프롬프트, 질문·선지 형식, greedy 생성/최대 새 토큰 2개, 파싱 정책은 저장 baseline과 동일합니다.
재학습·dev·test·최종 holdout 평가·자동 제출은 없습니다.

### 실행
1. baseline과 loss 비교를 실행한 같은 프로젝트에서 이 파일을 엽니다.
2. `LOSS_RUN_DIR`은 확인된 `LOSS-51ca09225d304a9a` 폴더로 설정했습니다.
3. `LOSS_MODE="answer_only"`로 설정했습니다. 동일 프로젝트에서 열면 별도 선택 없이 실행합니다. 경로가 다르면 PROJECT_DIR을 수정하세요.
4. Run All: 입력 검증 → P0 → P1 → P2 → P3 → P4 → P5 → 통합 비교표.

선택한 loss 조건의 학습·저장/재로드 검사가 완료돼 있어야 합니다. 모든 프롬프트가 동일 체크포인트를 공유합니다.
완료 조건은 재사용하고, 중단된 조건은 처음 문항부터 재평가합니다. 문항별 결과와 실제 프롬프트를 저장합니다.

### 예상 시간
새 학습 0회 / 검증 6회. 이번 answer_only 모델의 검증 실측은 441.34초(7분 21초)/500개입니다. **추론 합계 약 44분 8초 + 로딩·파일 검사 시간**이 기준 추정입니다.
새 후보 5개의 추론만 보면 약 36분 47초입니다. 이 파일은 공정한 기준 비교를 위해 P0도 다시 평가합니다.
이 시간은 선택한 학습640/추론672 모델의 실측을 기준으로 계산했습니다. 프롬프트 길이와 실행 환경에 따라 달라집니다.
**최초 Qwen3.5 baseline 전체 완료 시간이 없어 정확한 배수는 미확정**입니다. 전체 예상시간 / 실제 baseline 전체시간으로 비교하세요.

P5는 큰 글자→작은 글자 확인을 지시하지만 실제 시선·attention 순서를 강제하거나 crop/OCR을 수행하지는 않습니다.


## 1. 설정

In [1]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,csv
PROJECT_DIR=Path.cwd().resolve()
LOSS_RUN_DIR=PROJECT_DIR / "output/TASK-006-loss/LOSS-51ca09225d304a9a"  # 확인된 loss 실험
LOSS_MODE="answer_only"  # 검증 470/500, 94.0% 모델
ENV_PYTHON=PROJECT_DIR/"downloads/envs/TASK006_baseline_qwen35"/("Scripts/python.exe" if os.name=="nt" else "bin/python")
SESSION_TAG="prompt_P0_P5_answer_only_v1_1"
if LOSS_MODE not in ["full_text","answer_only"]:
    raise ValueError('LOSS_MODE에 선택할 "full_text" 또는 "answer_only"를 입력하세요.')
if LOSS_RUN_DIR is None:
    candidates=sorted(p.parent for p in (PROJECT_DIR/"output/TASK-006-loss").glob(f"LOSS-*/train_{LOSS_MODE}_status.json")
        if json.loads(p.read_text(encoding="utf-8")).get("state")=="completed")
    if len(candidates)!=1:
        print("완료 후보:",*[str(p) for p in candidates],sep="\n")
        raise RuntimeError("LOSS_RUN_DIR에 사용할 loss 실험 폴더를 지정하세요.")
    LOSS_RUN_DIR=candidates[0]
LOSS_RUN_DIR=Path(LOSS_RUN_DIR).resolve()
loss_cfg=json.loads((LOSS_RUN_DIR/"resolution_config.json").read_text(encoding="utf-8"))
BASELINE_RUN_DIR=Path(loss_cfg["baseline_dir"])
if not ENV_PYTHON.is_file():raise FileNotFoundError(ENV_PYTHON)
print("선택한 loss:",LOSS_MODE,"결과:",LOSS_RUN_DIR)


선택한 loss: answer_only 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a


## 2. 프롬프트 후보
P0는 저장된 baseline 함수를 그대로 사용합니다. P1~P5는 아래 instruction으로 마지막 지시만 교체합니다.

In [2]:
PROMPTS = {'P0': {'title': '기존 baseline', 'instruction': '정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.'}, 'P1': {'title': '이미지 근거', 'instruction': '이미지에서 실제로 확인할 수 있는 글자와 시각적 정보를 근거로 질문에 답하세요.\n이미지에 없는 내용을 일반 상식이나 추측으로 보충하지 마세요.\n질문에 가장 부합하는 선지 하나를 선택하세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.'}, 'P2': {'title': '문자·숫자 정밀 확인', 'instruction': '질문과 관련된 이미지 속 글자를 정확히 확인하세요.\n특히 숫자의 각 자리, 소수점, 가격, 날짜, 시간, 단위를 구분하고\n선지의 표기와 대조하여 정답을 선택하세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.'}, 'P3': {'title': '질문 조건·위치 관계', 'instruction': "질문이 요구하는 대상과 조건을 먼저 확인하세요.\n'아닌 것', '없는 것', '제외' 같은 부정 표현을 놓치지 말고,\n왼쪽·오른쪽·위·아래 등의 위치 표현은 질문에서 지정한 기준에 따라 해석하세요.\n이미지에서 해당 조건을 충족하는 선지 하나를 선택하세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요."}, 'P4': {'title': '선지 대조', 'instruction': '질문과 관련된 이미지 정보를 확인한 뒤 네 선지를 각각 대조하세요.\n선지 사이에서 달라지는 단어, 숫자, 단위와 위치를 확인하고,\n일부 내용만 일치하는 선지보다 질문의 조건 전체에 부합하는 선지를 선택하세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.'}, 'P5': {'title': '큰 글자에서 작은 글자로 확인', 'instruction': '질문이 요구하는 대상과 조건을 먼저 확인하세요.\n이미지의 큰 글자나 제목을 우선 읽고, 그 내용이 질문 및 네 선지와 연결되는지 대조하세요.\n큰 글자에서 선지에 해당하는 내용을 찾지 못하거나 정답을 결정할 근거가 부족하면,\n작은 글자·설명문·주석·가격·숫자·단위까지 확인 범위를 넓히세요.\n질문이 작은 글자나 특정 위치를 직접 지목하면 해당 부분을 우선 확인하세요.\n큰 글자와 일부 단어가 같다는 이유만으로 선택하지 말고 질문의 조건 전체에 부합하는 선지 하나를 고르세요.\n설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.'}}
PROMPT_ORDER=list(PROMPTS)
for name,p in PROMPTS.items():
    print(name,p["title"],"\n"+p["instruction"]+"\n")

P0 기존 baseline 
정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.

P1 이미지 근거 
이미지에서 실제로 확인할 수 있는 글자와 시각적 정보를 근거로 질문에 답하세요.
이미지에 없는 내용을 일반 상식이나 추측으로 보충하지 마세요.
질문에 가장 부합하는 선지 하나를 선택하세요.
설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.

P2 문자·숫자 정밀 확인 
질문과 관련된 이미지 속 글자를 정확히 확인하세요.
특히 숫자의 각 자리, 소수점, 가격, 날짜, 시간, 단위를 구분하고
선지의 표기와 대조하여 정답을 선택하세요.
설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.

P3 질문 조건·위치 관계 
질문이 요구하는 대상과 조건을 먼저 확인하세요.
'아닌 것', '없는 것', '제외' 같은 부정 표현을 놓치지 말고,
왼쪽·오른쪽·위·아래 등의 위치 표현은 질문에서 지정한 기준에 따라 해석하세요.
이미지에서 해당 조건을 충족하는 선지 하나를 선택하세요.
설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.

P4 선지 대조 
질문과 관련된 이미지 정보를 확인한 뒤 네 선지를 각각 대조하세요.
선지 사이에서 달라지는 단어, 숫자, 단위와 위치를 확인하고,
일부 내용만 일치하는 선지보다 질문의 조건 전체에 부합하는 선지를 선택하세요.
설명 없이 a, b, c, d 중 소문자 한 글자만 출력하세요.

P5 큰 글자에서 작은 글자로 확인 
질문이 요구하는 대상과 조건을 먼저 확인하세요.
이미지의 큰 글자나 제목을 우선 읽고, 그 내용이 질문 및 네 선지와 연결되는지 대조하세요.
큰 글자에서 선지에 해당하는 내용을 찾지 못하거나 정답을 결정할 근거가 부족하면,
작은 글자·설명문·주석·가격·숫자·단위까지 확인 범위를 넓히세요.
질문이 작은 글자나 특정 위치를 직접 지목하면 해당 부분을 우선 확인하세요.
큰 글자와 일부 단어가 같다는 이유만으로 선택하지 말고 질문의 조건 전체에 부합하는 선지 하나를 고

## 3. 로컬 실행 함수
한 GPU에서 한 조건씩 별도 프로세스로 실행합니다. Windows CRLF/LF 코드 해시 차이를 처리하며 실질적인 코드 변경은 차단합니다.

In [3]:
WORKER_SOURCE = r'''
import hashlib,importlib.util,json,os,sys,time,uuid
from pathlib import Path

def read(p):return json.loads(Path(p).read_text(encoding='utf-8'))
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for x in iter(lambda:f.read(4*1024*1024),b''):h.update(x)
    return h.hexdigest()
def write(p,obj):
    p=Path(p);t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8');t.replace(p)
def check_artifacts(record):
    if record['state']!='completed':raise RuntimeError('선택한 학습이 완료되지 않았습니다.')
    for name,h in record.get('artifacts',{}).items():
        p=Path(record['directory'])/name
        if not p.is_file() or sha(p)!=h:raise RuntimeError(f'완료된 학습 결과 변경/누락: {p}')

def context(cfg):
    base=Path(cfg['baseline_dir']);bc=read(base/'run_config.json');path=base/'task006_worker.py'
    data=path.read_bytes();raw=hashlib.sha256(data).hexdigest();lf=hashlib.sha256(data.replace(b'\r\n',b'\n')).hexdigest()
    if bc['worker_sha256'] not in (raw,lf) or bc['worker_sha256']!=cfg['baseline_worker_sha256']:raise RuntimeError('baseline 코드 변경')
    if sha(base/'run_config.json')!=cfg['baseline_config_sha256']:raise RuntimeError('baseline 설정 변경')
    spec=importlib.util.spec_from_file_location('saved_baseline',path);m=importlib.util.module_from_spec(spec);sys.modules[spec.name]=m;spec.loader.exec_module(m);m.block_network()
    record_path=Path(cfg['loss_dir'])/f"train_{cfg['loss_mode']}_status.json"
    if sha(record_path)!=cfg['train_record_sha256']:raise RuntimeError('선택 학습 기록 변경')
    record=read(record_path);check_artifacts(record)
    settings=read(Path(record['directory'])/'training_config.json')
    for key in ['seed','learning_rate','epochs','gradient_accumulation','model_id','revision']:
        if settings[key]!=bc[key]:raise RuntimeError(f'기준 학습과 불일치: {key}')
    if settings['pixel_budget']!=640**2 or settings['loss_mode']!=cfg['loss_mode']:raise RuntimeError('학습640/선택 loss를 확인하세요.')
    reload=read(Path(record['directory'])/'reload_check.json')
    if not reload.get('identical_outputs'):raise RuntimeError('저장/재로드 검증 미통과')
    loss_frozen=read(Path(cfg['loss_dir'])/'frozen_inputs.json')
    if loss_frozen['baseline_config_sha256']!=cfg['baseline_config_sha256']:raise RuntimeError('다른 baseline에서 생성된 loss 실험입니다.')
    _,valid,info=m.load_split(bc,base)
    if loss_frozen['data_manifest']!=info:raise RuntimeError('학습/검증 분할 불일치')
    return m,bc,record,valid,info

def status(cfg,name):
    p=Path(cfg['output_dir'])/f'{name}_status.json'
    if not p.exists():return {'state':'not_run'}
    rec=read(p)
    if rec['state']=='completed':check_artifacts(rec)
    return rec

def pair(a,b):
    x=a.merge(b,on='id',suffixes=('_P0','_new'),validate='one_to_one')
    if len(x)!=len(a) or len(x)!=len(b) or not (x.gold_P0==x.gold_new).all() or not (x.group_id_P0==x.group_id_new).all():raise RuntimeError('비교 대상 ID/정답/그룹 불일치')
    old=x.answer_P0==x.gold_P0;new=x.answer_new==x.gold_new
    x['transition']=['gain' if v and not u else 'loss' if u and not v else 'same_correct' if u else 'same_wrong' for u,v in zip(old,new)]
    return x,{'gain':int((~old&new).sum()),'loss':int((old&~new).sum()),'delta_pp':100*float(new.mean()-old.mean())}

def summary(cfg):
    import pandas as pd
    root=Path(cfg['output_dir']);rows=[];types=[];reference=None
    s=status(cfg,'P0')
    if s['state']=='completed':reference=pd.read_csv(Path(s['directory'])/'valid_predictions.csv',keep_default_na=False)
    for name in cfg['prompt_order']:
        s=status(cfg,name);r={'prompt':name,'title':cfg['prompts'][name]['title'],'status':s['state']}
        if s['state']=='completed':
            out=Path(s['directory']);r.update(read(out/'valid_metrics.json'));r['accuracy_pct']=100*r['accuracy'];r['parse_failure_pct']=100*r['parse_failure_rate']
            p=pd.read_csv(out/'valid_predictions.csv',keep_default_na=False)
            if reference is not None:
                x,stats=pair(reference,p);r.update(stats)
                x.to_csv(root/f'paired_P0_vs_{name}.csv',index=False,encoding='utf-8-sig')
                x[x.transition.isin(['gain','loss'])].to_csv(root/f'changed_P0_vs_{name}.csv',index=False,encoding='utf-8-sig')
            for kind,g in p.groupby('question_type'):
                types.append({'prompt':name,'question_type':kind,'n':len(g),'accuracy':float((g.answer==g.gold).mean())})
        else:r['error']=s.get('error','')
        rows.append(r)
    table=pd.DataFrame(rows);table.to_csv(root/'prompt_comparison.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(types,columns=['prompt','question_type','n','accuracy']).to_csv(root/'type_comparison.csv',index=False,encoding='utf-8-sig')
    write(root/'summary.json',{'results':rows,'loss_mode':cfg['loss_mode'],'adoption':'pending','test_used':False,'holdout_evaluated':False})
    (root/'PROJECT_STATUS_update.md').write_text('# 추론 프롬프트 비교\n\n동일 학습640 모델/추론672/고정 검증. 재학습 없음.\n\n'+table.to_string(index=False)+'\n\n채택/독립 검토 미완료.',encoding='utf-8')
    print(table.to_string(index=False),flush=True)
    return rows

def prepare(cfg):
    m,bc,record,valid,info=context(cfg);root=Path(cfg['output_dir'])
    assets=read(Path(cfg['baseline_dir'])/'model_assets.json')
    if assets['revision']!=bc['revision']:raise RuntimeError('revision 불일치')
    for name,meta in assets['files'].items():
        if sha(Path(bc['model_dir'])/name)!=meta['sha256']:raise RuntimeError('모델 파일 변경: '+name)
    valid[['id','group_id','question_type']].to_csv(root/'fixed_valid_ids.csv',index=False)
    write(root/'prepared.json',{'checkpoint':record['checkpoint'],'valid_n':len(valid),'data_manifest':info,
        'config_sha256':sha(root/'prompt_config.json'),'valid_ids_sha256':sha(root/'fixed_valid_ids.csv'),
        'model_stats':{n:[(Path(bc['model_dir'])/n).stat().st_size,(Path(bc['model_dir'])/n).stat().st_mtime_ns] for n in assets['files']}})
    write(root/'prompts.json',cfg['prompts']);print('준비 완료:',record['checkpoint'],'검증',len(valid),flush=True);summary(cfg)

def run(cfg,name):
    import torch
    from peft import PeftModel
    root=Path(cfg['output_dir']);frozen=read(root/'prepared.json')
    if sha(root/'prompt_config.json')!=frozen['config_sha256']:raise RuntimeError('준비 후 설정 변경')
    m,bc,record,valid,info=context(cfg)
    if sha(root/'fixed_valid_ids.csv')!=frozen['valid_ids_sha256'] or info!=frozen['data_manifest']:raise RuntimeError('검증 데이터 변경')
    for n,stats in frozen['model_stats'].items():
        p=Path(bc['model_dir'])/n
        if [p.stat().st_size,p.stat().st_mtime_ns]!=stats:raise RuntimeError('모델 파일 변경: 준비 단계 재실행 필요')
    previous=status(cfg,name)
    if previous['state'] in ['completed','blocked_oom']:
        print('기록 재사용:',name,previous['state'],flush=True);summary(cfg);return
    out=root/f'{name}_{time.strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}';out.mkdir()
    state_path=root/f'{name}_status.json';write(state_path,{'state':'running','directory':str(out)})
    settings=dict(bc);settings['pixel_budget']=672**2;settings['run_dir']=str(out)
    prompt=cfg['prompts'][name]
    original=m.build_mc_prompt
    def custom(question,a,b,c,d):
        return (f'{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n'+prompt['instruction'])
    if name!='P0':m.build_mc_prompt=custom
    sample=valid.iloc[0]
    rendered=m.build_mc_prompt(*(sample[k] for k in ['question','a','b','c','d']))
    if name=='P0' and rendered!=original(*(sample[k] for k in ['question','a','b','c','d'])):raise RuntimeError('P0 불일치')
    write(out/'prompt.json',{'name':name,**prompt,'system':m.SYSTEM_INSTRUCT,'example_user_message':rendered})
    write(out/'inference_config.json',settings)
    try:
        m.gpu_environment(settings,out);adapter=m.ModelAdapter(settings,out)
        adapter.model=PeftModel.from_pretrained(adapter.model,record['checkpoint'],local_files_only=True,is_trainable=False)
        probe=valid.iloc[0].to_dict();probe.pop('answer',None)
        inputs=adapter.encode(probe,training=False);input_tokens=int(inputs['input_ids'].shape[1]);adapter.generate(inputs);del inputs
        torch.cuda.synchronize()
        _,metrics=m.evaluate(adapter,valid,'valid',out)
        metrics.update({'checkpoint':record['checkpoint'],'train_resolution':640,'inference_resolution':672,
                        'loss_mode':cfg['loss_mode'],'warmup_samples':1,'first_input_tokens':input_tokens})
        write(out/'valid_metrics.json',metrics)
        files={str(p.relative_to(out)):sha(p) for p in out.rglob('*') if p.is_file()}
        write(state_path,{'state':'completed','directory':str(out),'artifacts':files})
    except torch.cuda.OutOfMemoryError as exc:
        write(state_path,{'state':'blocked_oom','directory':str(out),'error':str(exc)})
    except BaseException as exc:
        write(state_path,{'state':'failed_or_interrupted','directory':str(out),'error':str(exc)});raise
    finally:
        with open(root/'CHANGELOG.md','a',encoding='utf-8') as f:f.write(f"\n- {time.strftime('%Y-%m-%d %H:%M:%S')} {name}: {read(state_path)['state']}\n")
    summary(cfg)

if __name__=='__main__':
    cfg=read(sys.argv[1]);stage=sys.argv[2]
    if stage=='prepare':prepare(cfg)
    elif stage=='summary':summary(cfg)
    elif stage in cfg['prompt_order']:run(cfg,stage)
    else:raise ValueError(stage)

'''

In [4]:
def file_hash(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for x in iter(lambda:f.read(4*1024*1024),b""):h.update(x)
    return h.hexdigest()
base_cfg=json.loads((BASELINE_RUN_DIR/"run_config.json").read_text(encoding="utf-8"))
freeze=subprocess.check_output([str(ENV_PYTHON),"-m","pip","freeze"],text=True,encoding="utf-8")
expected=(BASELINE_RUN_DIR/"requirements.lock.txt").read_text(encoding="utf-8")
if sorted(freeze.splitlines())!=sorted(expected.splitlines()):raise RuntimeError("baseline 전용 패키지 환경이 변경됐습니다.")
CFG={"loss_dir":str(LOSS_RUN_DIR),"loss_mode":LOSS_MODE,"baseline_dir":str(BASELINE_RUN_DIR),
    "baseline_config_sha256":file_hash(BASELINE_RUN_DIR/"run_config.json"),
    "baseline_worker_sha256":base_cfg["worker_sha256"],
    "train_record_sha256":file_hash(LOSS_RUN_DIR/f"train_{LOSS_MODE}_status.json"),
    "prompts":PROMPTS,"prompt_order":PROMPT_ORDER,"session_tag":SESSION_TAG,
    "worker_sha256":hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
    "environment_sha256":hashlib.sha256(freeze.encode()).hexdigest()}
fingerprint=hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
OUTPUT_DIR=PROJECT_DIR/"output/TASK-006-prompts"/("PROMPT-"+fingerprint);OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CFG["output_dir"]=str(OUTPUT_DIR)
WORKER=OUTPUT_DIR/"prompt_worker.py";CONFIG=OUTPUT_DIR/"prompt_config.json"
compile(WORKER_SOURCE,str(WORKER),"exec");WORKER.write_bytes(WORKER_SOURCE.encode())
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_DIR/"requirements.lock.txt").write_text(freeze,encoding="utf-8")
def show_table():
    from IPython.display import display,Markdown,FileLink
    p=OUTPUT_DIR/"prompt_comparison.csv"
    if not p.exists():return
    with open(p,encoding="utf-8-sig",newline="") as f:rows=list(csv.DictReader(f))
    cols=[("prompt","조건"),("title","지시"),("status","상태"),("accuracy_pct","Accuracy %"),
          ("correct_n","정답"),("delta_pp","P0 대비 %p"),("gain","개선"),("loss","악화"),
          ("parse_failure_pct","파싱실패 %"),("seconds_per_sample","초/문항"),("peak_allocated_gib","VRAM GiB")]
    def fmt(v):
        if v is None or v=="":return "—"
        try:return f"{float(v):.3f}" if any(t in v for t in '.eE') else v
        except ValueError:return str(v).replace('|','/')
    text='| '+' | '.join(b for a,b in cols)+' |\n| '+' | '.join('---' for _ in cols)+' |\n'
    for row in rows:text+='| '+' | '.join(fmt(row.get(a)) for a,b in cols)+' |\n'
    display(Markdown(text));display(FileLink(str(p)));print("전체 결과:",OUTPUT_DIR)
def run_stage(stage):
    other_locks=[BASELINE_RUN_DIR/"running.lock",PROJECT_DIR/"output/TASK-006-loss/gpu_experiment.lock",
                 PROJECT_DIR/"output/TASK-006-resolution/gpu_experiment.lock",PROJECT_DIR/"output/TASK-006-training-resolution/gpu_experiment.lock"]
    for busy in other_locks:
        if busy.exists():raise RuntimeError(f"다른 실험 실행 잠금이 있습니다: {busy}. 종료 여부를 확인하세요.")
    lock=PROJECT_DIR/"output/TASK-006-prompts/gpu_experiment.lock"
    try:fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError(f"다른 프롬프트 실행 또는 잔여 잠금: {lock}. 프로세스가 종료됐을 때만 잔여 잠금을 삭제하세요.")
    proc=None
    try:
        with os.fdopen(fd,"w") as f:f.write(str(os.getpid()))
        env=os.environ.copy();env.update(PYTHONIOENCODING="utf-8",PYTHONUNBUFFERED="1")
        with open(OUTPUT_DIR/(stage+".log"),"a",encoding="utf-8") as log:
            proc=subprocess.Popen([str(ENV_PYTHON),"-u",str(WORKER),str(CONFIG),stage],stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,text=True,encoding="utf-8",errors="replace",env=env)
            for line in proc.stdout:print(line,end="");log.write(line);log.flush()
            if proc.wait()!=0:raise RuntimeError(f"{stage} 실패: 로그 {OUTPUT_DIR/(stage+'.log')}")
    except BaseException:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=10)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
        sp=OUTPUT_DIR/(stage+"_status.json")
        if sp.exists():
            rec=json.loads(sp.read_text(encoding="utf-8"))
            if rec["state"]=="running":
                rec["state"]="interrupted";sp.write_text(json.dumps(rec,ensure_ascii=False,indent=2),encoding="utf-8")
        raise
    finally:lock.unlink(missing_ok=True)
    show_table()
print("결과:",OUTPUT_DIR)


결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 4. 모델·분할·입력 검증
기존 checkpoint와 이미지/분할 파일을 확인합니다. 준비 시간은 추론 지표에 포함하지 않습니다. 다른 GPU 학습이 실행 중이면 종료 후 시작하세요.

In [5]:
run_stage("prepare")

준비 완료: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\train_answer_only_20260922_152814_5e73bca1\adapter_epoch1 검증 500
prompt            title  status error
    P0      기존 baseline not_run      
    P1           이미지 근거 not_run      
    P2      문자·숫자 정밀 확인 not_run      
    P3      질문 조건·위치 관계 not_run      
    P4            선지 대조 not_run      
    P5 큰 글자에서 작은 글자로 확인 not_run      


| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | not_run | — | — | — | — | — | — | — | — |
| P1 | 이미지 근거 | not_run | — | — | — | — | — | — | — | — |
| P2 | 문자·숫자 정밀 확인 | not_run | — | — | — | — | — | — | — | — |
| P3 | 질문 조건·위치 관계 | not_run | — | — | — | — | — | — | — | — |
| P4 | 선지 대조 | not_run | — | — | — | — | — | — | — | — |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 5. P0 — 기존 baseline

In [6]:
run_stage("P0")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 16:37:04.858000 29348 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<09:12,  1.37it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | not_run | — | — | — | — | — | — | — | — |
| P2 | 문자·숫자 정밀 확인 | not_run | — | — | — | — | — | — | — | — |
| P3 | 질문 조건·위치 관계 | not_run | — | — | — | — | — | — | — | — |
| P4 | 선지 대조 | not_run | — | — | — | — | — | — | — | — |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 6. P1 — 이미지 근거

In [7]:
run_stage("P1")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 16:44:03.075000 20176 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:34,  1.47it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469.000 | -0.200 | 2.000 | 3.000 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | not_run | — | — | — | — | — | — | — | — |
| P3 | 질문 조건·위치 관계 | not_run | — | — | — | — | — | — | — | — |
| P4 | 선지 대조 | not_run | — | — | — | — | — | — | — | — |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 7. P2 — 문자·숫자 정밀 확인

In [8]:
run_stage("P2")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 16:51:13.104000 16056 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:36,  1.47it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469.000 | -0.200 | 2.000 | 3.000 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | completed | 94.000 | 470.000 | 0.000 | 3.000 | 3.000 | 0.000 | 0.813 | 7.658 |
| P3 | 질문 조건·위치 관계 | not_run | — | — | — | — | — | — | — | — |
| P4 | 선지 대조 | not_run | — | — | — | — | — | — | — | — |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 8. P3 — 질문 조건·위치 관계

In [9]:
run_stage("P3")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 16:58:27.164000 30344 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:46,  1.44it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469.000 | -0.200 | 2.000 | 3.000 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | completed | 94.000 | 470.000 | 0.000 | 3.000 | 3.000 | 0.000 | 0.813 | 7.658 |
| P3 | 질문 조건·위치 관계 | completed | 93.800 | 469.000 | -0.200 | 1.000 | 2.000 | 0.000 | 0.820 | 7.665 |
| P4 | 선지 대조 | not_run | — | — | — | — | — | — | — | — |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 9. P4 — 선지 대조

In [10]:
run_stage("P4")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 17:05:45.525000 29100 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:59,  1.41it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470.000 | 0.000 | 0.000 | 0.000 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469.000 | -0.200 | 2.000 | 3.000 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | completed | 94.000 | 470.000 | 0.000 | 3.000 | 3.000 | 0.000 | 0.813 | 7.658 |
| P3 | 질문 조건·위치 관계 | completed | 93.800 | 469.000 | -0.200 | 1.000 | 2.000 | 0.000 | 0.820 | 7.665 |
| P4 | 선지 대조 | completed | 93.400 | 467.000 | -0.600 | 0.000 | 3.000 | 0.000 | 0.865 | 7.660 |
| P5 | 큰 글자에서 작은 글자로 확인 | not_run | — | — | — | — | — | — | — | — |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 10. P5 — 큰 글자에서 작은 글자로 확인

In [11]:
run_stage("P5")

{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'os': 'Windows-10-10.0.26200-SP0', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'NVIDIA GeForce RTX 5060 Ti', 'vram_gib': 15.92828369140625, 'packages': {'transformers': '5.8.0', 'peft': '0.18.1', 'bitsandbytes': '0.49.2', 'accelerate': '1.12.0'}, 'network': 'blocked; local model files only'}
W0922 17:13:27.417000 13476 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d

Loading weights:   0%|          | 2/760 [00:01<08:55,  1.41it/s]C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\envs\TASK006_baseline_qwen35\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: F

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470 | 0.000 | 0 | 0 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469 | -0.200 | 2 | 3 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | completed | 94.000 | 470 | 0.000 | 3 | 3 | 0.000 | 0.813 | 7.658 |
| P3 | 질문 조건·위치 관계 | completed | 93.800 | 469 | -0.200 | 1 | 2 | 0.000 | 0.820 | 7.665 |
| P4 | 선지 대조 | completed | 93.400 | 467 | -0.600 | 0 | 3 | 0.000 | 0.865 | 7.660 |
| P5 | 큰 글자에서 작은 글자로 확인 | completed | 94.200 | 471 | 0.200 | 2 | 1 | 0.000 | 1.001 | 7.687 |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 11. 통합 결과

In [12]:
run_stage("summary")

prompt            title    status   n  correct_n  accuracy  parse_failure_rate  fallback_usage_rate    seconds  seconds_per_sample  peak_allocated_gib  peak_reserved_gib                                                                                                                                checkpoint  train_resolution  inference_resolution   loss_mode  warmup_samples  first_input_tokens  accuracy_pct  parse_failure_pct  gain  loss  delta_pp
    P0      기존 baseline completed 500        470     0.940                 0.0                  0.0 390.937611            0.781875            7.652091          11.333984 C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-loss\LOSS-51ca09225d304a9a\train_answer_only_20260922_152814_5e73bca1\adapter_epoch1               640                   672 answer_only               1                 544          94.0                0.0     0     0       0.0
    P1           이미지 근거 completed 500        469     0.938                 0.0                  0.

| 조건 | 지시 | 상태 | Accuracy % | 정답 | P0 대비 %p | 개선 | 악화 | 파싱실패 % | 초/문항 | VRAM GiB |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| P0 | 기존 baseline | completed | 94.000 | 470 | 0.000 | 0 | 0 | 0.000 | 0.782 | 7.652 |
| P1 | 이미지 근거 | completed | 93.800 | 469 | -0.200 | 2 | 3 | 0.000 | 0.807 | 7.658 |
| P2 | 문자·숫자 정밀 확인 | completed | 94.000 | 470 | 0.000 | 3 | 3 | 0.000 | 0.813 | 7.658 |
| P3 | 질문 조건·위치 관계 | completed | 93.800 | 469 | -0.200 | 1 | 2 | 0.000 | 0.820 | 7.665 |
| P4 | 선지 대조 | completed | 93.400 | 467 | -0.600 | 0 | 3 | 0.000 | 0.865 | 7.660 |
| P5 | 큰 글자에서 작은 글자로 확인 | completed | 94.200 | 471 | 0.200 | 2 | 1 | 0.000 | 1.001 | 7.687 |


C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22\prompt_comparison.csv

전체 결과: C:\Users\SSAFY\Desktop\AI2_Challenge\output\TASK-006-prompts\PROMPT-4af48e1c8b465b22


## 결과 파일

- **prompt_comparison.csv**: 정확도·정답 수·P0 대비 개선/악화·파싱 실패·시간·VRAM. 매 조건마다 갱신합니다.
- `P번호_실행시각/valid_predictions.csv`: 문항별 원출력/예측/정답/그룹/이미지 처리 정보.
- `P번호_실행시각/prompt.json`: 실제 시스템 지시·사용자 지시 예시.
- `changed_P0_vs_P번호.csv`: 기준 대비 개선/악화 문항.
- `type_comparison.csv`: 기존 정규식 기반 임시 문항 유형별 지표.
- `summary.json`, `PROJECT_STATUS_update.md`, `CHANGELOG.md`: 결과 및 반영용 기록. 공식 기준본은 수정하지 않습니다.

선택 모델의 보고된 P0 기준은 470/500=94.0%, 파싱 실패 0%입니다. 이번 P0는 동일 checkpoint로 다시 평가하며, 94.0%를 결과 표에 미리 채우지 않습니다. 총점이 다르면 평가 로그를 확인하세요. 과거 학습384 모델의 93.2%는 이번 기준이 아닙니다.
P5가 큰 글자에 과도하게 의존해 작은 글자 문항을 놓치는지도 오답에서 확인하세요. 동일한 프롬프트를 모든 문항에 적용하며 유형별 자동 분기는 하지 않습니다.
프롬프트 효과는 선택한 checkpoint에 조건부입니다. 학습 프롬프트까지 바꾸려면 별도 재학습 비교가 필요합니다.
새 프롬프트 선택은 자동 확정하지 않습니다. 검증셋 반복 사용과 작은 점수 차이에 유의하세요.
문법·CPU 비교 로직은 검사했지만 작성 환경에서 실제 GPU 실행은 하지 않았습니다.
